# 🫀 실험 1′ — Icentia11k 대규모 확증 · **모든 산출물은 Google Drive에 남는다**

**MedKOS / `notebooks/exp1_icentia_rhythm_scale.ipynb`** · 퀘스트 `ailab-2026-0015`

---

## ⚠️ 이 노트북이 "실험 1"이 아니라 "실험 1′"인 이유

`mit-bih/PAPER.md`(선행 트랙)를 확인한 결과, 원래 계획한 **실험 1(보상성 휴지 → S 구제)은
이미 완료**되어 있습니다.

| 계획했던 arm | 이미 한 것 | 결과 (SVDB 73환자, 환자매크로 F1) |
|---|---|---|
| A. 형태만 | `B2` CNN(raw) | **0.164** |
| B. +RR | `B3` CNN+RR | **0.484** — Δ**+0.320** [+0.246,+0.396] Bonferroni 통과 ★ |
| C. +보상지수 물리 | `B4`(RHYTHM innovation) | 0.534 — B4−B3 Δ+0.050 **[−0.022,+0.122] 비유의** |

**"리듬을 넣어라"는 확증됐고, "보상지수를 별도 축으로"는 판정이 안 났습니다.**
그 이유는 방법이 아니라 검정력입니다 — `PAPER.md` §8.4 가설3 · §10 Future Work 1:

> 73환자로는 0.05 크기 효과의 검정력이 부족하다. 배제하려면 **수백 명 규모**가 필요하다.

**이 노트북은 새 방법을 만들지 않습니다. 검정력을 올려 "비유의"를 확정 판정으로 바꿉니다.**
CI 반폭 0.073(73명) → **0.031(400명) / 0.021(900명)**.

---

## 사전등록 (실행 전 고정 — 결과 보고 바꾸지 않기)

```
주지표 : 환자단위 매크로 F1 (N/S/V)        ← micro 금지 (PAPER §8.3, #232 지배)
비교   : B3R − B3, 대응 부트스트랩 B=2000, 환자 단위 리샘플
판정   : 95% CI가 0을 포함하지 않으면 '확증', 포함하면 '기각(효과 없음)'
저울   : seed 확률 평균 앙상블              ← 단일 seed는 ±0.15 요동 (PAPER §0)
금지   : 테스트 환자로 어떤 선택도 하지 않는다
```

---

## 📁 모든 것이 Drive에 남습니다

```
MyDrive/MedKOS/ecg-model/            ← 이 프로젝트의 단일 루트
├── README.md                        레이아웃 설명 (자동 생성)
├── lib/medkos_run.py                아티팩트 헬퍼 (자동 기록 → 다음 노트북이 재사용)
├── data/                            코호트 캐시 (공유 — 재다운로드 안 함)
│   └── icentia_n600_m15.npz
├── runs/<타임스탬프>_<실험id>/       ← 실행마다 폴더 하나
│   ├── config.json                  설정 전체 (재현용)
│   ├── manifest.json                시각·환경·GPU·라이브러리 버전
│   ├── log.txt                      모든 출력
│   ├── cohort_stats.json
│   ├── figures/physics_precheck.png
│   ├── arms/B3R/{weights.keras, probs.npy}
│   └── result.json                  ← ingest_run.py 입력
├── registry.jsonl                   ★ 모든 실행 1줄씩 = 실험 대장
└── ASSETS.md                        기존 흩어진 자산 위치 등록
```

**`registry.jsonl` 이 핵심입니다.** "어떤 실험을 무슨 설정으로 돌렸고 결과가 뭐였나"가
파일 하나에 계속 쌓입니다. 지금까지 흩어져 있던 문제의 근본 해결책입니다.

**이어하기 지원**: 같은 `RUN_ID`로 다시 실행하면 이미 끝난 arm은 건너뜁니다.
Colab이 끊겨도 처음부터 다시 안 해도 됩니다.

---

## 실행 방법
1. 런타임 → 런타임 유형 변경 → **T4 GPU**
2. 위에서부터 순서대로 실행
3. **CELL 3(스모크 테스트)에서 한 번 멈추고** 주석 심볼을 확인하세요

**예상 시간** — `QUICK=True`: 15~25분 / `QUICK=False`: 2시간 내외


In [ ]:
# CELL 1 — Drive 마운트 + 프로젝트 레이아웃 + 아티팩트 헬퍼
!pip -q install wfdb

import os, sys, json, time, shutil, platform, textwrap

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab이 아님 — 로컬 폴더로 대체:", e)
    DRIVE_ROOT = "/content"

PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")   # ← 이 프로젝트의 단일 루트
for sub in ("lib", "data", "runs", "figures"):
    os.makedirs(os.path.join(PROJECT, sub), exist_ok=True)

# ── 헬퍼 소스: Drive의 lib/medkos_run.py 로 기록해 다음 노트북이 재사용 ──
HELPER = r"""
import os, sys, json, time, platform, subprocess

class MedKOSRun:
    \"\"\"실행 하나의 모든 산출물을 Drive에 남기는 아티팩트 스토어.

    run = MedKOSRun("exp1p_icentia", config, project=PROJECT)
    run.log("...")            # 화면 + log.txt
    run.save_json("a", obj);  run.save_npy("b", arr);  run.save_fig("c", fig)
    run.save_model(m, "B3R"); run.load_npy("b")
    run.done("B3R")           # 이 arm 끝났나?  (이어하기)
    run.finish(result)        # result.json + registry.jsonl 한 줄 append
    \"\"\"

    def __init__(self, exp_id, config, project, run_id=None):
        self.project = project
        self.exp_id = exp_id
        self.run_id = run_id or f"{time.strftime('%Y%m%dT%H%M')}_{exp_id}"
        self.dir = os.path.join(project, "runs", self.run_id)
        self.data_dir = os.path.join(project, "data")
        for d in (self.dir, self.data_dir,
                  os.path.join(self.dir, "figures"), os.path.join(self.dir, "arms")):
            os.makedirs(d, exist_ok=True)
        self.t0 = time.time()
        self.save_json("config", config)
        self.save_json("manifest", {
            "run_id": self.run_id, "exp_id": exp_id,
            "started": time.strftime("%Y-%m-%dT%H:%M:%S"),
            "python": sys.version.split()[0], "platform": platform.platform(),
            "gpu": self._gpu(), "packages": self._pkgs(),
        })
        self.log(f"RUN {self.run_id}")
        self.log(f"  → {self.dir}")

    # ---------- 환경 ----------
    def _gpu(self):
        try:
            return subprocess.check_output(
                ["nvidia-smi", "--query-gpu=name,memory.total",
                 "--format=csv,noheader"], text=True).strip()
        except Exception:
            return "none"

    def _pkgs(self):
        out = {}
        for m in ("numpy", "tensorflow", "wfdb", "sklearn"):
            try:
                out[m] = __import__(m).__version__
            except Exception:
                out[m] = "n/a"
        return out

    # ---------- 경로 ----------
    def path(self, *parts):
        p = os.path.join(self.dir, *parts)
        os.makedirs(os.path.dirname(p), exist_ok=True)
        return p

    def data(self, name):
        return os.path.join(self.data_dir, name)

    # ---------- 기록 ----------
    def log(self, msg=""):
        print(msg)
        with open(os.path.join(self.dir, "log.txt"), "a") as f:
            f.write(f"[{time.strftime('%H:%M:%S')}] {msg}\n")

    def save_json(self, name, obj):
        p = self.path(f"{name}.json")
        with open(p, "w") as f:
            json.dump(obj, f, ensure_ascii=False, indent=2, default=str)
        return p

    def save_npy(self, name, arr):
        import numpy as np
        p = self.path(f"{name}.npy"); np.save(p, arr); return p

    def load_npy(self, name):
        import numpy as np
        p = os.path.join(self.dir, f"{name}.npy")
        return np.load(p) if os.path.exists(p) else None

    def save_fig(self, name, fig=None):
        import matplotlib.pyplot as plt
        p = self.path("figures", f"{name}.png")
        (fig or plt.gcf()).savefig(p, dpi=140, bbox_inches="tight")
        return p

    def save_model(self, model, arm):
        p = self.path("arms", arm, "weights.keras")
        try:
            model.save(p)
        except Exception as e:
            self.log(f"  (모델 저장 실패 {arm}: {e})")
        return p

    # ---------- 이어하기 ----------
    def done(self, arm):
        return os.path.exists(os.path.join(self.dir, "arms", arm, "probs.npy"))

    def save_arm(self, arm, probs):
        import numpy as np
        p = self.path("arms", arm, "probs.npy"); np.save(p, probs); return p

    def load_arm(self, arm):
        import numpy as np
        p = os.path.join(self.dir, "arms", arm, "probs.npy")
        return np.load(p) if os.path.exists(p) else None

    # ---------- 마무리 ----------
    def finish(self, result):
        result = dict(result)
        result["run_id"] = self.run_id
        result["elapsed_sec"] = round(time.time() - self.t0, 1)
        self.save_json("result", result)
        line = {k: result.get(k) for k in
                ("run_id", "exp_id", "date", "metric", "value", "passed")}
        line["exp_id"] = self.exp_id
        line["summary"] = result.get("summary", "")
        line["dir"] = self.dir
        with open(os.path.join(self.project, "registry.jsonl"), "a") as f:
            f.write(json.dumps(line, ensure_ascii=False) + "\n")
        self.log(f"FINISH {self.run_id}  ({result['elapsed_sec']:.0f}s)")
        self.log(f"  registry.jsonl 에 1줄 추가됨")
        return result
"""

LIB = os.path.join(PROJECT, "lib", "medkos_run.py")
with open(LIB, "w") as f:
    f.write(HELPER.replace('\\"\\"\\"', '"""'))
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun            # noqa

# ── README (레이아웃 설명, 한 번만) ──
README = os.path.join(PROJECT, "README.md")
if not os.path.exists(README):
    with open(README, "w") as f:
        f.write(textwrap.dedent("""\
        # MedKOS / ecg-model — 단일유도 ECG 진단 모델 프로젝트 루트

        | 폴더 | 내용 |
        |---|---|
        | `lib/` | 공용 헬퍼 (`medkos_run.py`) — 노트북이 자동 갱신 |
        | `data/` | 코호트 캐시. **실행 간 공유**되므로 재다운로드하지 않음 |
        | `runs/<시각>_<실험id>/` | 실행 하나의 전부: config·manifest·log·figures·arms·result |
        | `registry.jsonl` | **실험 대장** — 모든 실행이 한 줄씩. 여기부터 보면 됨 |
        | `ASSETS.md` | 기존에 흩어져 있던 자산 위치 등록 |

        ## 다음 노트북에서 재사용하는 법
        ```python
        import sys; sys.path.insert(0, "/content/drive/MyDrive/MedKOS/ecg-model/lib")
        from medkos_run import MedKOSRun
        run = MedKOSRun("exp2_xxx", config, project="/content/drive/MyDrive/MedKOS/ecg-model")
        ```
        """))

# ── ASSETS.md (기존 흩어진 자산 등록, 한 번만) ──
ASSETS = os.path.join(PROJECT, "ASSETS.md")
if not os.path.exists(ASSETS):
    with open(ASSETS, "w") as f:
        f.write(textwrap.dedent("""\
        # 기존 자산 위치 (이동하지 말고 여기에 '등록'만 한다)

        > 옮기면 예전 노트북 경로가 깨진다. **위치를 기록**해서 찾을 수 있게만 한다.

        ## 선행 연구 트랙 (mit-bih / SVEB) — repo `mit-bih/`, `PAPER.md` 수준
        | 자산 | Drive 위치 |
        |---|---|
        | MIT-BIH 특징 캐시 | `MyDrive/mitbih/mamba_data.npz` (약 205 MB) |
        | SVDB 특징 캐시 | `MyDrive/mitbih/svdb_data.npz` (약 447 MB) |
        | SVDB 특징 폴더 | `MyDrive/mitbih/svdb_feats/` |
        | 전처리·벤치 스크립트 | `MyDrive/mitbih/svdb_prep.py`, `svdb_bench.py`, `svdb_rhythm.py`, `svdb_labels.py` |

        ## AI랩 학습 트랙
        | 자산 | Drive 위치 |
        |---|---|
        | 1주차 체크포인트 | `MyDrive/MedKOS/ailab/week01/` |
        | 콘텐츠 파생물 | `MyDrive/MedKOS/content/` |

        ## 이 프로젝트 (ecg-model)
        모든 신규 산출물은 `MyDrive/MedKOS/ecg-model/runs/` 아래에만 쌓는다.
        """))

print("✅ 프로젝트 루트:", PROJECT)
print("   lib/medkos_run.py · README.md · ASSETS.md 준비 완료")

In [ ]:
# CELL 2 — 설정 + 실행 시작 (여기 값만 바꾸면 됩니다)
import numpy as np, random
from collections import Counter

# ══════════════════════════════════════════════════════
QUICK  = True          # True: 빠른 1회전 / False: 본 실험
RUN_ID = None          # 이어하기: 이전 run 폴더명을 그대로 넣으면 끝난 arm은 건너뜀
# ══════════════════════════════════════════════════════

if QUICK:
    N_PATIENTS, MINUTES, N_SEEDS, EPOCHS = 120, 12, 2, 8
else:
    N_PATIENTS, MINUTES, N_SEEDS, EPOCHS = 600, 15, 3, 12

CONFIG = dict(
    exp="exp1p_icentia_rhythm_scale", quest="ailab-2026-0015",
    hypothesis="B3R - B3 (보상지수 축)의 95% CI가 0을 벗어나는가",
    dataset="icentia11k-continuous-ecg/1.0",
    n_patients=N_PATIENTS, minutes=MINUTES, n_seeds=N_SEEDS, epochs=EPOCHS,
    fs=250, w_pre=100, w_post=150, classes=["N", "S", "V"],
    test_frac=0.35, seed0=20260731, boot=2000, quick=QUICK,
)
FS, W_PRE, W_POST = CONFIG["fs"], CONFIG["w_pre"], CONFIG["w_post"]
CLASSES, TEST_FRAC, SEED0 = CONFIG["classes"], CONFIG["test_frac"], CONFIG["seed0"]
random.seed(SEED0); np.random.seed(SEED0)

run = MedKOSRun("exp1p_icentia", CONFIG, project=PROJECT, run_id=RUN_ID)
run.log(f"환자 {N_PATIENTS}명 · {MINUTES}분 · seed {N_SEEDS} · epoch {EPOCHS}")
run.log(f"코호트 캐시 디렉토리: {run.data_dir}")

### CELL 3 — 경로 자동 탐색 + 스모크 테스트 (★ 여기서 한 번 멈추세요)

Icentia11k는 환자당 약 70분짜리 세그먼트 50개로 쪼개져 있고, 파일이 `p00/p00000/p00000_s00`
처럼 **2단 하위 폴더** 안에 있습니다.

> ⚠️ **wfdb의 함정**: `rdrecord("p00/p00000/p00000_s00", pn_dir=DB)` 처럼 부르면
> wfdb가 **앞의 폴더 경로를 버리고** `DB/p00000_s00.hea` 를 찾아 404가 납니다.
> **하위 폴더는 반드시 `pn_dir` 쪽에** 넣어야 합니다:
> `rdrecord("p00000_s00", pn_dir=f"{DB}/p00/p00000")`

경로를 추측하지 않고 **PhysioNet의 `RECORDS` 파일을 직접 읽어** 실제 경로를 씁니다.
`sampto=`로 앞부분만 읽으므로 환자당 수 MB만 내려옵니다.

**주석 심볼이 `N/S/V/Q`가 아니면 CELL 4의 `SYMBOL_MAP`을 고쳐야 합니다.**

In [ ]:
# CELL 3 — RECORDS로 실제 경로 확보 + 주석 심볼 확인
import wfdb, re

DB_CANDIDATES = ["icentia11k-continuous-ecg/1.0", "icentia11k-continuous-ecg"]

def split_pn(rel, db):
    """'p00/p00000/p00000_s00' → (pn_dir, record_name).
    ★ wfdb는 하위 폴더를 record_name이 아니라 pn_dir에 넣어야 한다."""
    d, b = os.path.split(rel.strip().rstrip("/"))
    return (f"{db}/{d}" if d else db), b

# ── RECORDS 목록 (Drive 캐시 — 55만 줄이라 한 번만 받는다) ──
REC_CACHE = run.data("icentia_RECORDS.json")
DB, RECS = None, None
if os.path.exists(REC_CACHE):
    blob = json.load(open(REC_CACHE)); DB, RECS = blob["db"], blob["records"]
    run.log(f"RECORDS 캐시 재사용: {len(RECS):,}개  (db={DB})")
else:
    for db in DB_CANDIDATES:
        try:
            run.log(f"RECORDS 다운로드 시도: {db} …")
            RECS = list(wfdb.get_record_list(db))
            if RECS:
                DB = db
                json.dump({"db": DB, "records": RECS}, open(REC_CACHE, "w"))
                run.log(f"✅ RECORDS {len(RECS):,}개 확보 → 캐시 저장")
                break
        except Exception as e:
            run.log(f"   실패: {type(e).__name__}: {str(e)[:90]}")

if not RECS:
    run.log("❌ RECORDS를 못 받았습니다. PhysioNet 페이지의 Files 탭에서 구조를 확인하세요.")
else:
    run.log(f"앞 5개 항목: {RECS[:5]}")

# ── 환자 인덱스 PAT: pid → [레코드 상대경로들] ────────────────────────
# RECORDS는 데이터셋마다 '레코드 목록'일 수도, '폴더 목록'일 수도 있다.
# 0단/1단/2단 중첩 어느 쪽이든 동작하도록, 필요한 수만큼만 내려간다.
PAT, _WALKED = {}, set()
PAT_CACHE = run.data("icentia_PAT.json")
if os.path.exists(PAT_CACHE):
    try:
        _b = json.load(open(PAT_CACHE))
        PAT = {int(k): v for k, v in _b["pat"].items()}; _WALKED = set(_b["walked"])
        run.log(f"환자 인덱스 캐시 재사용: {len(PAT):,}명")
    except Exception as e:
        run.log(f"인덱스 캐시 무시: {e}")

def _list(sub=""):
    return list(wfdb.get_record_list(DB if not sub else f"{DB}/{sub.rstrip('/')}"))

def _index(entries, prefix=""):
    n = 0
    for e in entries:
        rel = f"{prefix}{str(e).strip()}".rstrip("/")
        m = re.search(r"p(\d{5})_s(\d+)", rel)
        if m:
            PAT.setdefault(int(m.group(1)), []).append(rel); n += 1
    return n

def ensure_patients(target):
    """PAT에 target명 이상 모일 때까지 폴더를 내려간다(필요한 만큼만)."""
    if len(PAT) >= target:
        return
    if _index(RECS or []) == 0:                       # 최상위가 폴더 목록인 경우
        for d in (RECS or []):
            if len(PAT) >= target:
                break
            d = str(d).strip().rstrip("/")
            if d in _WALKED:
                continue
            _WALKED.add(d)
            try:
                sub = _list(d)
            except Exception:
                continue
            if _index(sub, prefix=f"{d}/") == 0:       # 한 단계 더 (2단 구조)
                for d2 in sub:
                    if len(PAT) >= target:
                        break
                    d2 = str(d2).strip().rstrip("/")
                    try:
                        _index(_list(f"{d}/{d2}"), prefix=f"{d}/{d2}/")
                    except Exception:
                        continue
    for k in PAT:
        PAT[k].sort()
    try:                                               # 다음 실행에서 재탐색 안 하도록
        json.dump({"pat": {str(k): v for k, v in PAT.items()},
                   "walked": sorted(_WALKED)}, open(PAT_CACHE, "w"))
    except Exception:
        pass

ensure_patients(6)                                     # 스모크는 6명이면 충분
run.log(f"\n환자 {len(PAT):,}명 인덱싱(스모크용) · 환자당 세그먼트 "
        f"{int(np.median([len(v) for v in PAT.values()])) if PAT else 0}개")
run.log("  ※ 본 수집은 CELL 4에서 필요한 만큼 더 내려갑니다")

# ── 스모크 리드: 실제 2명만 읽어 fs·심볼 확인 ──
probe, ok = {}, 0
for pid in sorted(PAT)[:6]:
    rel = PAT[pid][0]
    pn, base = split_pn(rel, DB)
    try:
        n = FS * 60
        rec = wfdb.rdrecord(base, pn_dir=pn, sampfrom=0, sampto=n)
        ann = wfdb.rdann(base, "atr", pn_dir=pn, sampfrom=0, sampto=n)
        sym = Counter(ann.symbol).most_common()
        aux = Counter([a for a in getattr(ann, "aux_note", []) if a and a.strip()]).most_common()[:6]
        run.log(f"✅ {rel}\n   pn_dir={pn}  record={base}")
        run.log(f"   fs={rec.fs} sig={rec.sig_name} shape={rec.p_signal.shape}")
        run.log(f"   심볼 {sym}")
        run.log(f"   리듬 {aux}")
        probe[rel] = {"pn_dir": pn, "record": base, "fs": rec.fs,
                      "sig": rec.sig_name, "symbols": sym, "rhythm": aux}
        ok += 1
        if ok >= 2:
            break
    except Exception as e:
        run.log(f"❌ {rel}: {type(e).__name__}: {str(e)[:110]}")

run.save_json("smoke_probe", {"db": DB, "n_records": len(RECS or []),
                              "n_patients": len(PAT), "probe": probe})
run.log("\n확인: ① fs=250 ② 심볼에 N/S/V/Q ③ 여전히 ❌면 probe 출력을 보고 알려주세요")

In [ ]:
# CELL 4 — 코호트 구축 (Drive 캐시 공유 · 재실행 시 다운로드 생략)
# CELL 3 출력의 심볼이 다르면 여기를 고치세요.
SYMBOL_MAP = {"N": 0, "n": 0, "·": 0,
              "S": 1, "A": 1, "a": 1, "J": 1,
              "V": 2, "E": 2,
              "Q": 3, "?": 3}

def robust_scale(x):
    med = np.median(x)
    iqr = np.percentile(x, 75) - np.percentile(x, 25)
    return ((x - med) / (iqr + 1e-6)).astype("float32")

def load_patient(pid, minutes=MINUTES, seg_idx=0):
    """PAT 인덱스(RECORDS 유래)의 실제 경로로 읽는다 — 경로를 만들어내지 않는다."""
    rel = PAT[pid][seg_idx]
    pn, base = split_pn(rel, DB)
    n = FS * 60 * minutes
    rec = wfdb.rdrecord(base, pn_dir=pn, sampfrom=0, sampto=n)
    ann = wfdb.rdann(base, "atr", pn_dir=pn, sampfrom=0, sampto=n)
    sig = robust_scale(rec.p_signal[:, 0].astype("float64"))
    keep = [i for i, s in enumerate(ann.symbol) if s in SYMBOL_MAP]
    if len(keep) < 60:
        return None
    samp = np.asarray(ann.sample)[keep]
    lab = np.array([SYMBOL_MAP[ann.symbol[i]] for i in keep], dtype="int64")
    X, y, t = [], [], []
    for k in range(len(samp)):
        c = samp[k]
        if c - W_PRE < 0 or c + W_POST > len(sig):
            continue
        X.append(sig[c - W_PRE:c + W_POST]); y.append(lab[k]); t.append(c / FS)
    if len(y) < 60:
        return None
    return np.array(X, "float16"), np.array(y), np.array(t, "float64")


CACHE = run.data(f"icentia_n{N_PATIENTS}_m{MINUTES}.npz")   # ← runs/가 아니라 data/ (공유)
if os.path.exists(CACHE):
    d = np.load(CACHE)
    Xb, yb, tb, pidb = d["X"], d["y"], d["t"], d["pid"]
    run.log(f"캐시 재사용: {CACHE}")
else:
    ensure_patients(int(N_PATIENTS * 1.4))        # ★ 실패분을 감안해 여유 있게 인덱싱
    run.log(f"인덱싱된 환자 {len(PAT):,}명 → {N_PATIENTS}명 목표로 수집 시작")
    Xs, ys, ts, ps = [], [], [], []
    got, t0 = 0, time.time()
    candidates = sorted(PAT)                      # RECORDS에 실제로 있는 환자만
    fails = 0
    for pid in candidates:
        if got >= N_PATIENTS:
            break
        try:
            out = load_patient(pid)
        except Exception as e:
            fails += 1
            if fails <= 3:
                run.log(f"  [skip p{pid:05d}] {type(e).__name__}: {str(e)[:80]}")
            continue
        if out is None:
            continue
        X, y, t = out
        Xs.append(X); ys.append(y); ts.append(t)
        ps.append(np.full(len(y), pid, dtype="int32")); got += 1
        if got % 20 == 0:
            el = time.time() - t0
            run.log(f"  {got}/{N_PATIENTS}명 ({el:.0f}s · 예상 총 {el/got*N_PATIENTS/60:.1f}분)")
    Xb = np.concatenate(Xs); yb = np.concatenate(ys)
    tb = np.concatenate(ts); pidb = np.concatenate(ps)
    np.savez_compressed(CACHE, X=Xb, y=yb, t=tb, pid=pidb)
    run.log(f"캐시 저장: {CACHE}")

stats = {"n_patients": int(len(np.unique(pidb))), "n_beats": int(len(yb)),
         "class_counts": {n: int((yb == c).sum()) for c, n in enumerate(["N", "S", "V", "Q"])}}
run.save_json("cohort_stats", stats)
run.log(f"\n환자 {stats['n_patients']}명 · 비트 {stats['n_beats']:,}개")
for c, n in enumerate(["N", "S", "V", "Q"]):
    m = yb == c
    run.log(f"  {n}: {m.sum():8,} ({m.mean():6.2%})  보유환자 {len(np.unique(pidb[m])):4d}명")

# ══════════ 안전장치 — 여기서 걸리면 아래 셀을 돌리지 마세요 ══════════
STOP = []
if stats["n_patients"] < 40:
    STOP.append(f"환자 {stats['n_patients']}명뿐 — 부트스트랩 CI가 무의미해집니다 "
                f"(테스트 환자 ≥20 필요). RECORDS 인덱싱이 덜 됐을 가능성.")
for c, n in enumerate(["N", "S", "V"]):
    if (yb == c).sum() == 0:
        STOP.append(f"{n} 클래스 비트가 0개 — SYMBOL_MAP이 실제 심볼과 다르거나 코호트가 너무 작음")
if STOP:
    run.log("\n" + "⛔" * 30)
    for s in STOP:
        run.log("⛔ " + s)
    run.log("⛔ 이 상태로 학습하면 결과가 전부 무의미합니다. 여기서 멈추고 알려주세요.")
    run.log("⛔" * 30)
else:
    run.log("\n✅ 코호트 점검 통과 — 다음 셀로 진행하세요")

### CELL 5 — 리듬 특징: 단순 RR vs RHYTHM innovation

**이 둘의 차이가 이 실험의 전부**입니다.

| | 내용 | 의미 |
|---|---|---|
| `RR` (B3) | `rr_pre/ref`, `rr_post/ref`, `ref` | **정보만** 준다 |
| `RHY` (B3R) | RR 3개 + `pre_innov`, `post_innov`, **`comp = pre+post`**, `mad_ratio` | **법칙까지** 준다 |

`comp`(보상지수)의 생리 — `PAPER.md` §4.2:
* **PVC**: 심실 기원 → 동방결절 리셋 못함 → **완전 보상성 휴지** → `comp ≈ 0`
* **PAC**: 심방 기원 → 동방결절 리셋 → **불완전 보상성 휴지** → `comp < 0`

innovation은 환자별 인과 EWMA 잔차를 MAD로 정규화하고 tanh로 유계화 → **무차원** →
진폭·기기·샘플링에 불변 → 교차DB 전이가 원리적으로 가능.

> ⚠️ 이 셀은 CELL 4 직후 **한 번만** 실행 (끝에서 Q비트를 걸러냅니다).

In [ ]:
# CELL 5 — RHYTHM innovation (PAPER.md §4.2 재구현)
ALPHA, K_REF = 0.3, 10

def rhythm_features(t, alpha=ALPHA):
    n = len(t); rr = np.diff(t)
    if len(rr) < 5:
        return None, None
    pred = np.empty_like(rr); acc = rr[0]
    for i in range(len(rr)):                       # 인과적 EWMA (과거만)
        pred[i] = acc; acc = alpha * rr[i] + (1 - alpha) * acc
    resid = rr - pred
    mad_g = np.median(np.abs(resid - np.median(resid))) + 1e-9
    mad_l = np.array([np.median(np.abs(resid[max(0, i-25):i+26]
                                       - np.median(resid[max(0, i-25):i+26])))
                      for i in range(len(resid))])
    scale = np.maximum(mad_l, 0.2 * mad_g) + 1e-9
    innov = np.tanh(resid / scale / 3.0)           # (-1,1) 유계
    rr_ref = np.maximum([np.median(rr[max(0, i-K_REF):i+K_REF+1])
                         for i in range(len(rr))], 1e-3)
    RR, RHY = [], []
    for k in range(n):
        i_pre, i_post = k - 1, k
        if i_pre < 0 or i_post >= len(rr):
            RR.append([1., 1., .8]); RHY.append([1., 1., .8, 0, 0, 0, 1.]); continue
        ref = rr_ref[i_pre]; pi, po = innov[i_pre], innov[i_post]
        RR.append([rr[i_pre]/ref, rr[i_post]/ref, ref])
        RHY.append([rr[i_pre]/ref, rr[i_post]/ref, ref,
                    pi, po, pi + po, scale[i_pre]/mad_g])   # ★ comp = pi+po
    return np.array(RR, "float32"), np.array(RHY, "float32")

F_RR = np.zeros((len(yb), 3), "float32"); F_RHY = np.zeros((len(yb), 7), "float32")
for pid in np.unique(pidb):
    m = pidb == pid
    a, b = rhythm_features(tb[m])
    if a is not None:
        F_RR[m], F_RHY[m] = a, b

keep = yb < 3                                       # Q 제외
Xb, yb, tb, pidb = Xb[keep], yb[keep], tb[keep], pidb[keep]
F_RR, F_RHY = F_RR[keep], F_RHY[keep]
run.save_npy("features_RR", F_RR); run.save_npy("features_RHY", F_RHY)
run.log(f"비트 {len(yb):,} · RR{F_RR.shape} · RHY{F_RHY.shape} → Drive 저장")

### CELL 6 — ★ 물리 사전점검 (학습 전에 반드시)

**모델을 돌리기 전에 법칙이 이 데이터에서 성립하는지 봅니다.** S와 V의 보상지수가
겹쳐 있으면 학습을 아무리 해도 이 축은 정보를 못 줍니다 — 2분 만에 알 수 있습니다.

In [ ]:
# CELL 6 — 보상지수가 S와 V를 실제로 가르는가 (Cohen's d)
import matplotlib.pyplot as plt

comp, prem = F_RHY[:, 5], F_RR[:, 0]
ect = prem < 0.85
pre = {"overall": {}, "ectopic": {}}

run.log("=== 전체 ===")
for c in range(3):
    m = yb == c
    if m.sum() == 0:
        pre["overall"][CLASSES[c]] = {"n": 0}
        run.log(f"  {CLASSES[c]}: n=0  ⛔ 이 클래스가 없습니다 — SYMBOL_MAP 또는 코호트 크기 확인")
        continue
    pre["overall"][CLASSES[c]] = {"n": int(m.sum()),
                                  "prem_median": float(np.median(prem[m])),
                                  "comp_median": float(np.median(comp[m]))}
    run.log(f"  {CLASSES[c]}: n={m.sum():7,} 조기성={np.median(prem[m]):.3f}"
            f"  보상지수={np.median(comp[m]):+.3f}")

run.log("\n=== 조기박동(prem<0.85)만 — 여기서 갈려야 함 ===")
for c in (1, 2):
    m = ect & (yb == c)
    if m.sum() > 10:
        q1, q3 = np.percentile(comp[m], [25, 75])
        pre["ectopic"][CLASSES[c]] = {"n": int(m.sum()),
                                      "comp_median": float(np.median(comp[m])),
                                      "iqr": [float(q1), float(q3)]}
        run.log(f"  {CLASSES[c]}: n={m.sum():6,} 보상지수={np.median(comp[m]):+.3f}"
                f" IQR=[{q1:+.3f},{q3:+.3f}]")

mS, mV = ect & (yb == 1), ect & (yb == 2)
if mS.sum() > 10 and mV.sum() > 10:
    d = float((comp[mV].mean() - comp[mS].mean()) /
              np.sqrt((comp[mV].var() + comp[mS].var()) / 2 + 1e-9))
    pre["cohens_d_SV"] = d
    run.log(f"\n  ▶ 보상지수의 S↔V 분리 Cohen's d = {d:+.3f}")
    run.log("     |d|>0.8 큼 / 0.5~0.8 중간 / <0.2 거의 없음")
    run.log("     ※ |d|<0.2면 학습해도 안 오릅니다 — 여기서 판단 가능")
run.save_json("physics_precheck", pre)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for c, col in zip(range(3), ["#888", "#c0392b", "#1c4780"]):
    ax[0].hist(comp[yb == c], bins=60, range=(-2, 2), alpha=.55, density=True,
               label=CLASSES[c], color=col)
ax[0].set_title("compensatory index — all beats"); ax[0].legend()
for c, col in zip((1, 2), ["#c0392b", "#1c4780"]):
    m = ect & (yb == c)
    if m.sum() > 10:
        ax[1].hist(comp[m], bins=50, range=(-2, 2), alpha=.6, density=True,
                   label=CLASSES[c], color=col)
ax[1].axvline(0, ls="--", c="k", lw=1)
ax[1].set_title("compensatory index — ectopic only (S vs V)"); ax[1].legend()
plt.tight_layout()
run.log("그림 저장: " + run.save_fig("physics_precheck", fig))
plt.show()

In [ ]:
# CELL 7 — 환자 단위 분할 + 모델 정의
import tensorflow as tf
from tensorflow.keras import layers, models

pats = np.unique(pidb); rng = np.random.RandomState(SEED0); rng.shuffle(pats)
n_te = int(len(pats) * TEST_FRAC)
te_p = set(pats[:n_te].tolist())
is_te = np.isin(pidb, list(te_p))
split = {"train_patients": int(len(pats) - n_te), "test_patients": int(n_te),
         "test_patient_ids": sorted(int(p) for p in te_p)}
run.save_json("split", split)
run.log(f"학습 {split['train_patients']}명 / 테스트 {split['test_patients']}명 (환자 겹침 없음)")
if n_te < 20:
    run.log(f"⛔ 테스트 환자가 {n_te}명뿐입니다. 대응 부트스트랩은 '환자'를 리샘플하므로")
    run.log(f"⛔ 환자가 1명이면 CI 폭이 0이 되고, 20명 미만이면 CI를 신뢰할 수 없습니다.")
for name, m in (("train", ~is_te), ("test", is_te)):
    run.log(f"  {name}: {m.sum():7,}비트  " +
            "  ".join(f"{CLASSES[c]}={int(((yb==c)&m).sum()):,}" for c in range(3)))

def auto_weights(y, beta=0.9999):                  # PAPER §4.4 (학습셋에서만 유도)
    w = {c: (1 - beta) / (1 - beta ** max((y == c).sum(), 1)) for c in range(3)}
    return {c: float(v / w[0]) for c, v in w.items()}

CW = auto_weights(yb[~is_te])
run.log(f"클래스 가중치: {({CLASSES[c]: round(v,2) for c,v in CW.items()})}")

def build(n_feat, seed):
    tf.keras.utils.set_random_seed(seed)
    sig_in = layers.Input((W_PRE + W_POST, 1), name="beat")
    x = sig_in
    for f, k in ((32, 7), (64, 5), (128, 3)):
        x = layers.Conv1D(f, k, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x); x = layers.MaxPooling1D(2)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation="relu")(x)
    ins = [sig_in]
    if n_feat > 0:
        f_in = layers.Input((n_feat,), name="feat")
        g = layers.Dense(32, activation="relu")(f_in)
        g = layers.Dense(32, activation="relu")(g)
        x = layers.Concatenate()([x, g]); ins = [sig_in, f_in]
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation="relu")(x)
    m = models.Model(ins, layers.Dense(3, activation="softmax")(x))
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy")
    return m

ARMS = {"B2": None, "B3": F_RR, "B3R": F_RHY}
ARM_DESC = {"B2": "형태만", "B3": "형태+RR", "B3R": "형태+RHYTHM(보상지수)"}
Xf = Xb.astype("float32")[..., None]
run.log(f"ARM: {ARM_DESC}")

In [ ]:
# CELL 8 — 학습 (arm × seed) · arm 단위 이어하기 · 확률을 Drive에 저장
def feat_norm(F, trm):
    mu, sd = F[trm].mean(0), F[trm].std(0) + 1e-6
    return ((F - mu) / sd).astype("float32")

probs, t0 = {}, time.time()
for arm, F in ARMS.items():
    cached = run.load_arm(arm)
    if cached is not None:
        probs[arm] = cached
        run.log(f"⏭  {arm} 이미 완료 — 건너뜀 (probs.npy 재사용)")
        continue
    acc = np.zeros((int(is_te.sum()), 3), "float64")
    for s in range(N_SEEDS):
        if F is None:
            xtr, xte = Xf[~is_te], Xf[is_te]
        else:
            Fn = feat_norm(F, ~is_te)
            xtr, xte = [Xf[~is_te], Fn[~is_te]], [Xf[is_te], Fn[is_te]]
        m = build(0 if F is None else F.shape[1], SEED0 + s)
        m.fit(xtr, yb[~is_te], epochs=EPOCHS, batch_size=512,
              class_weight=CW, verbose=0)
        acc += m.predict(xte, batch_size=1024, verbose=0)
        if s == 0:
            run.save_model(m, arm)                       # seed0 가중치만 보관
        run.log(f"  {arm} seed {s+1}/{N_SEEDS} 완료 ({time.time()-t0:.0f}s)")
        tf.keras.backend.clear_session()
    probs[arm] = acc / N_SEEDS
    run.save_arm(arm, probs[arm])                        # ← 즉시 Drive 저장(이어하기)
run.log(f"학습 총 {time.time()-t0:.0f}초")

### CELL 9 — 평가: 환자단위 매크로 F1 + 대응 부트스트랩

`PAPER.md` §5.1 프로토콜 그대로입니다.

* **환자단위 매크로** — 비트를 다 모으는 micro는 S가 많은 소수 환자에게 지배됩니다
  (MIT-BIH DS2에서 #232 하나가 S의 75.2%). 환자별 F1의 평균을 씁니다.
* **대응 부트스트랩** — 환자마다 난이도가 달라서, 대응(paired)이 아니면 arm 간 차이가
  환자 분산에 묻힙니다.

In [ ]:
# CELL 9 — 평가 + 사전등록 가설 판정
from sklearn.metrics import f1_score, confusion_matrix

y_te, pid_te = yb[is_te], pidb[is_te]
upa = np.unique(pid_te)

def per_patient_macro(pred):
    out = []
    for p in upa:
        m = pid_te == p
        present = np.unique(np.concatenate([y_te[m], pred[m]]))
        out.append(f1_score(y_te[m], pred[m], labels=present,
                            average="macro", zero_division=0))
    return np.array(out)

res = {}
for arm, pr in probs.items():
    pred = pr.argmax(1); pp = per_patient_macro(pred)
    res[arm] = {"pp": pp, "macro": float(pp.mean()),
                "f1": f1_score(y_te, pred, average=None, labels=[0, 1, 2], zero_division=0),
                "cm": confusion_matrix(y_te, pred, labels=[0, 1, 2]).tolist()}

run.log("=" * 74)
run.log(f"{'ARM':<26}{'환자매크로 F1':>16}{'N':>9}{'S':>9}{'V':>9}")
run.log("=" * 74)
for arm, r in res.items():
    f = r["f1"]
    run.log(f"{arm+' '+ARM_DESC[arm]:<26}{r['macro']:>16.4f}"
            f"{f[0]:>9.3f}{f[1]:>9.3f}{f[2]:>9.3f}")
run.log("=" * 74)

def paired_boot(a, b, B=CONFIG["boot"], seed=SEED0):
    rs = np.random.RandomState(seed); d = a - b
    boot = d[rs.randint(0, len(d), size=(B, len(d)))].mean(1)
    return float(d.mean()), float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))

boots = {}
run.log(f"\n{'비교':<16}{'Δ':>10}{'95% CI':>24}   판정")
run.log("-" * 62)
for hi, lo in (("B3", "B2"), ("B3R", "B3"), ("B3R", "B2")):
    d, l, u = paired_boot(res[hi]["pp"], res[lo]["pp"])
    sig = bool(l > 0 or u < 0)
    boots[f"{hi}-{lo}"] = {"delta": d, "ci": [l, u], "significant": sig}
    run.log(f"{hi+' − '+lo:<16}{d:>+10.4f}   [{l:+.4f}, {u:+.4f}]   {'유의 ★' if sig else '비유의'}")
run.log("-" * 62)

kb = boots["B3R-B3"]; half = (kb["ci"][1] - kb["ci"][0]) / 2
run.log(f"\n▶ 사전등록 주가설 — B3R − B3 (보상지수 축의 순수 기여)")
run.log(f"   테스트 환자 {len(upa)}명")
run.log(f"   Δ = {kb['delta']:+.4f}   95% CI [{kb['ci'][0]:+.4f}, {kb['ci'][1]:+.4f}]")
run.log(f"   CI 반폭 = {half:.4f}   (SVDB 73환자 때 0.073)")

# ★ 유의성과 '방향'은 다르다. 음의 유의를 확증으로 읽으면 안 된다.
if len(upa) < 20:
    verdict = (f"판정 불가 — 테스트 환자 {len(upa)}명뿐. 부트스트랩이 같은 환자만 "
               f"리샘플해 CI가 인위적으로 좁아진다(환자 1명이면 폭 0). 환자 수를 늘려 재실행")
elif kb["significant"] and kb["delta"] > 0:
    verdict = "확증 — 보상지수 축은 단순 RR 위에 유의한 정보를 더한다"
elif kb["significant"] and kb["delta"] < 0:
    verdict = "역효과 — 보상지수 축을 넣으면 오히려 나빠진다(특징 중복·차원 증가 의심)"
elif half < 0.03:
    verdict = "기각(확정) — 단순 RR로 정보 포화. PAPER §8.4 가설1 지지"
else:
    verdict = "미결 — 검정력 부족. QUICK=False + N_PATIENTS 900으로 재실행"
run.log(f"   → {verdict}")
if len(upa) < 20:
    run.log("   ⛔ 이 수치는 보고하지 마세요. 코호트를 먼저 키워야 합니다.")
run.save_json("evaluation", {
    "arms": {a: {"macro_f1": r["macro"],
                 "f1_per_class": {CLASSES[i]: float(r["f1"][i]) for i in range(3)},
                 "confusion": r["cm"]} for a, r in res.items()},
    "bootstrap": boots, "ci_halfwidth": half, "verdict": verdict})

In [ ]:
# CELL 10 — 마무리: result.json + registry.jsonl + repo 반영 명령
result = {
    "week": 1, "exp_id": "exp1p_icentia", "quest": "ailab-2026-0015",
    "task": "Icentia11k 대규모 확증 — 보상지수 축 (실험1′)",
    "split": "inter", "metric": "macro_f1",
    "value": round(res["B3R"]["macro"], 4),
    "passed": bool(boots["B3R-B3"]["significant"]),
    "date": time.strftime("%Y-%m-%d"),
    "n_patients": stats["n_patients"], "n_test_patients": int(len(upa)),
    "n_beats": int(len(yb)),
    "arms": {a: round(r["macro"], 4) for a, r in res.items()},
    "bootstrap": boots, "ci_halfwidth": round(half, 4), "verdict": verdict,
    "summary": (f"B3R-B3 Δ{kb['delta']:+.4f} CI[{kb['ci'][0]:+.4f},{kb['ci'][1]:+.4f}] "
                f"n_pat={stats['n_patients']} → {verdict.split(' —')[0]}"),
}
result = run.finish(result)

# 로컬에도 복사(ingest_run.py 에 바로 넣기 편하게)
import shutil
shutil.copy(os.path.join(run.dir, "result.json"), "/content/result.json")

print(json.dumps({k: v for k, v in result.items() if k != "bootstrap"},
                 ensure_ascii=False, indent=2))
print(f"""
────────────────────────────────────────────────────────────────
📁 이번 실행 전체:  {run.dir}
📒 실험 대장:       {os.path.join(PROJECT, 'registry.jsonl')}

repo에 실행 로그로 박기 (result.json 을 내려받아 repo 루트에 두고)

  python pipelines/ingest_run.py --results result.json \\
      --notebook notebooks/exp1_icentia_rhythm_scale.ipynb \\
      --quest ailab-2026-0015 --step "exp1p-icentia-scale" \\
      --note "{result['summary']}"
────────────────────────────────────────────────────────────────""")

In [ ]:
# CELL 11 — (선택) 실험 대장 훑어보기: 지금까지 돌린 모든 실행
reg = os.path.join(PROJECT, "registry.jsonl")
if os.path.exists(reg):
    rows = [json.loads(l) for l in open(reg) if l.strip()]
    print(f"총 {len(rows)}개 실행\n" + "="*100)
    for r in rows[-20:]:
        print(f"{r.get('run_id','?'):<28} {str(r.get('value')):>8}  "
              f"{'PASS' if r.get('passed') else '    '}  {r.get('summary','')[:52]}")
else:
    print("아직 실행 기록이 없습니다.")

---

## 결과 읽는 법

| B3R − B3 의 95% CI | 결론 | 다음 |
|---|---|---|
| **0을 포함하지 않고 +** | 보상지수 축 실재. `PAPER.md` §8.4 가설3(표본 한계)이 원인이었음 | 이 축을 Physics Layer 첫 제약으로 승격 → 퀘스트 실험 2 |
| **0 포함 · 반폭 < 0.03** | 단순 RR로 **정보 포화**(가설1). 확정적 음성 결과 | 물리를 *특징*으로 짜내지 말고 **loss·구조 제약**으로. 또는 유도 확장(실험 9·10) |
| **0 포함 · 반폭 > 0.05** | 아직 검정력 부족 | `QUICK=False` + `N_PATIENTS=900`으로 재실행 (캐시가 있어 다운로드는 재사용) |

**세 경우 모두 진행 가능합니다.** 음성이어도 `PAPER.md`의 세 가설 중 하나를 배제하므로
그대로 논문 문장이 됩니다.

## 이 노트북이 Drive에 남기는 자산

| 자산 | 위치 | 왜 중요한가 |
|---|---|---|
| **실험 대장** | `ecg-model/registry.jsonl` | 흩어짐 문제의 근본 해결 — 모든 실행이 한 줄씩 |
| 코호트 캐시 | `ecg-model/data/` | 실행 간 공유. 900명으로 늘려도 기존 분량은 재사용 |
| 실행 폴더 | `ecg-model/runs/<시각>_<id>/` | config·manifest·log·figures·arms·result 전부 |
| arm 확률 | `runs/.../arms/<arm>/probs.npy` | **재학습 없이 평가만 다시** 가능 |
| 공용 헬퍼 | `ecg-model/lib/medkos_run.py` | 다음 노트북은 `from medkos_run import MedKOSRun` 한 줄 |

## 하지 않은 것 (정직하게)
- Icentia11k 라벨은 기술자 1인 전수 판독 — MIT-BIH의 2인 심장전문의 합의와 **품질 등급이 다릅니다**. 절대 성능을 MIT-BIH·SVDB 수치와 직접 비교하지 마세요. **arm 간 차이만** 비교합니다.
- 환자당 앞부분 N분만 사용합니다(다운로드 비용). 기록 후반의 리듬 변화는 보지 않습니다.
- 임계값 최적화 없음(argmax). 동작점 실험은 별도입니다.
- 이 노트북은 기존 Drive 폴더(`MyDrive/mitbih/`, `MedKOS/ailab/`)를 **건드리지 않습니다.** 위치만 `ASSETS.md`에 등록합니다.
